In [77]:
from pathlib import Path
import numpy as np
import torch
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, BatchSampler
import random
from itertools import islice
import math

ROOT = Path("../data/processed/classification").resolve()

transforms = v2.Compose([
  v2.ToImage(), 
  v2.Resize((64,64)),
])
train_ds = ImageFolder(ROOT / "train", transform=transforms)
dev_ds = ImageFolder(ROOT / "dev", transform=transforms)

print(len(dev_ds.targets))
print(dev_ds.targets[:10])


104016
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [78]:
class BalancedBatchSampler(BatchSampler):
  def __init__(self, targets, batch_size, g):

    self.targets = np.array(targets, dtype=np.int64)
    self.batch_size = batch_size
    self.classes = np.unique(self.targets).tolist()
    self.n_classes = len(self.classes)
    self.g = g

    print(f"{[type(c) for c in self.classes]}, {self.n_classes}")

    self.samples_per_class = batch_size // self.n_classes
    print(f"samples per class: {self.samples_per_class}")

    # TODO: assert self.target is 1d, batch//   

    # targets has to be a np.array for this to be safe, otherwise is wrong
    self.class_indices = {
      c: np.nonzero(self.targets == c)[0]
      for c in self.classes
    }
    # print(self.class_indices)

     # Minority class size
    self.minority_size = min(
      len(v) for v in self.class_indices.values()
    )
    # identify minority class
    self.minority_class = min(
      self.class_indices,
      key=lambda c: len(self.class_indices[c])
    )
    self.minority_indices = (
      self.class_indices[self.minority_class]
    )
    # print([self.class_indices[3]])
    print(f"minority size: {self.minority_size}")
    print(f"minority class: {self.minority_class}")

    # TODO: fix that to not drop the remaining samples from minority class
    # self.n_batches = self.minority_size // self.samples_per_class
    # fix: twra exw akeraia batches + 1, kai to last batch tha parei oti menei apo
    # to minority class + samples_per_class apo ola ta alla
    # e.g. periseuoun 2 minority, kai samples_per_class=4, last_batch: [4maj, 4maj, 2min, 4maj]
    self.n_batches = math.ceil(self.minority_size / self.samples_per_class)

    print(f"batches: {self.n_batches}")

  def __iter__(self):

    shuffled = {} # same as class_indices , but shuffled
    for c, idxs in self.class_indices.items():
      perm = torch.randperm(len(idxs), generator=self.g)
      shuffled[c] = idxs.copy()[perm]
      print(f"{c} : {idxs[:6]} | {shuffled[c][:6]}")

    for batch_idx in range(self.n_batches):
      batch = []

      start = batch_idx*self.samples_per_class
      end = start + self.samples_per_class # for the last batch and minority class this is > len(idxes) but is ok, i just keep the remaining

      for c in self.classes:
        batch.extend(shuffled[c][start:end].tolist())
        print(f"{batch_idx} for class {c} added {shuffled[c][start:end]}")

      bperm = torch.randperm(len(batch), generator=self.g)
      batch = [batch[i] for i in bperm]
      yield batch

  def __len__(self):
    return self.n_batches

In [79]:
generator = torch.Generator().manual_seed(123456789)
# sampler = BalancedBatchSampler(targets=dev_ds.targets, batch_size=24,g=generator)
sampler = BalancedBatchSampler(targets=train_ds.targets, batch_size=240, g=generator)

# print(next(iter(sampler)))
# print(len(next(iter(sampler))))

# for batch in islice(sampler, 3):
#   print(f"batch: {len(batch)}")
# for i, batch in enumerate(sampler):
#   print(f"{i}: {len(batch)}")

[<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>], 6
samples per class: 40
minority size: 2066
minority class: 4
batches: 52


In [80]:
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=0)
# dev_loader = DataLoader(dev_ds, batch_sampler=sampler, num_workers=0)

# for epoch in range(2):
for (images, labels) in islice(train_loader, 2):
  print(images.shape, labels)
  # print("==================================================")

0 : [0 1 2 3 4 5] | [ 68328 414394 198296  14862   2558 306365]
1 : [421520 421521 421522 421523 421524 421525] | [591226 586327 592474 454878 637640 498094]
2 : [663307 663308 663309 663310 663311 663312] | [790504 756697 663402 714470 690789 916031]
3 : [924685 924686 924687 924688 924689 924690] | [927441 925769 927228 925659 928830 925558]
4 : [928964 928965 928966 928967 928968 928969] | [930374 929128 930464 929093 929343 930211]
5 : [931030 931031 931032 931033 931034 931035] | [935287 932177 933204 931322 935370 934161]
0 for class 0 added [ 68328 414394 198296  14862   2558 306365 327414 303790 283698  98536
 234267 118233 131137 132337 150030 326667 259699 288090 208830  67107
 235546 242460 142047 184426 357668 217040 284407 208667  90239 264991
 396808 314458 370919  38772 257563  46837  16110  59643 182778 147492]
0 for class 1 added [591226 586327 592474 454878 637640 498094 657716 660858 589850 487756
 505080 489796 619624 628441 441691 461626 648425 660592 499404 595577

In [ ]:
class BalancedBatchSampler(Sampler):
  """
  Creates batches with equal number of samples per class.

  Example:
      batch_size = 252
      num_classes = 6
      -> 42 samples per class per batch
  """

  def __init__(self, labels, batch_size, shuffle=True):
    self.labels = np.array(labels)
    self.classes = np.unique(self.labels)

    self.num_classes = len(self.classes)

    assert batch_size % self.num_classes == 0, (
      "Batch size must be divisible by number of classes"
    )

    self.samples_per_class = batch_size // self.num_classes
    self.batch_size = batch_size
    self.shuffle = shuffle

    # Store indices for each class
    self.class_indices = {
      c: np.where(self.labels == c)[0].tolist()
      for c in self.classes
    }

    # Minority class determines epoch length
    self.min_class_count = min(len(v) for v in self.class_indices.values())

    # Number of batches so minority class is fully used
    self.num_batches = (
      self.min_class_count // self.samples_per_class
    )

  def __iter__(self):

    # Shuffle class indices every epoch
    pools = {}

    for c, idxs in self.class_indices.items():
      idxs = idxs.copy()

      if self.shuffle:
        random.shuffle(idxs)

      pools[c] = idxs

    class_ptrs = {c: 0 for c in self.classes}

    for _ in range(self.num_batches):

      batch = []

      for c in self.classes:

        idxs = pools[c]
        ptr = class_ptrs[c]

        # If not enough samples remain:
        if ptr + self.samples_per_class > len(idxs):

          # reshuffle and restart majority classes
          if self.shuffle:
            random.shuffle(idxs)

          ptr = 0

        selected = idxs[ptr:ptr+self.samples_per_class]

        batch.extend(selected)

        class_ptrs[c] = ptr + self.samples_per_class

      if self.shuffle:
        random.shuffle(batch)

      yield batch

  def __len__(self):
    return self.num_batches

In [ ]:

ROOT = Path("../data/processed/classification").resolve()

transforms = v2.Compose([
  v2.ToImage(), 
  v2.Resize((64,64)),
])
train_ds = ImageFolder(ROOT / "train", transform=transforms)
dev_ds = ImageFolder(ROOT / "dev", transform=transforms)

print(len(train_ds.targets))

sampler = BalancedBatchSampler(
    labels=train_ds.targets,
    batch_size=252
)

loader = DataLoader(
  dev_ds,
  batch_sampler=sampler,
  num_workers=4
)

for (imgs, targets) in loader:
  print(imgs.shape)

936118
